## Построение фонетического транскриптора (преобразования графема → фонема) для русского языка.

Используем парсер из Лабораторной работы 1. Он извлекает признаки:

original — исходное написание слова, берём из атрибута original тега word. Если атрибут пустой, оставляем пустую строку;
nucleus / phrasal_stress — фразовое ударение слова. Ставится на ядро синтагмы, если указано тегом nucleus="2". Если ядро не указано, ударение ставится на последнее слово синтагмы;
pause_len — длительность паузы после слова, значение из тега pause. Если пауза меньше 150 мс, считаем её отсутствующей (-1), если больше 800 мс, ограничиваем 800 мс;
phrasal_stress — слово является ядром синтагмы (фразовое ударение);
is_capital — слово с заглавной буквы;
length — длина слова;
fonetic_words_before / fonetic_words_after — количество слов с фонетическим выделением до и после текущего слова в предложении;
word_num — индекс слова в предложении;
sentence_len — количество слов в предложении;
prev_word / next_word — предыдущие и следующие слова (после нормализации);
sintagma_num — номер синтагмы с учетом пауз;
has_pause / pause — наличие паузы и её длительность;
genesys - род и одушевленность слова

Импорты

In [20]:
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import torch
from tqdm import tqdm
import pickle
import json
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

Парсер XML-файлов корпуса.

Задачи:
• пройти по XML-документу;
• извлечь структуру предложений;
• извлечь слова, буквы и разметку аллофонов (если они есть);
• построить полезные дополнительные признаки (позиция слова, паузы и т.д.).

In [21]:
class Parser:
    def parse_text(self, xml_root): # Собираем все слова всех предложений
        self.sintagma_num = 0
        parsed_words = []
        for child in xml_root:
            if child.tag == "sentence":
                parsed_words.extend(self.parse_sentence(child))
        return parsed_words

    def parse_sentence(self, xml_sentence): # Извлекаем слова, паузы, порядок слов и фонетические признаки.
        sentence_words = []
        words_count = 0
        fonetic_words_count = 0
        word = {}
        for i in range(len(xml_sentence)):
            child = xml_sentence[i]
            # Слово
            if child.tag == "word":
                # Читаем внутреннюю структуру <word>
                word_info = self.parse_word(child)
                word.update(word_info)
                # сколько фонетически выделенных слов было до
                word["fonetic_words_before"] = fonetic_words_count
                # порядковый номер слова в предложении
                word["word_num"] = words_count + 1
                # паузы и синтагмы — парсер обновит при встрече <pause>
                word["has_pause"] = False
                word["sintagma_num"] = self.sintagma_num

                words_count += 1
                fonetic_words_count += self.check_if_fonetic(child)

                sentence_words.append(word)
                word = {}

            elif child.tag == "content":
                pass
            # Пауза
            elif child.tag == "pause":
                # Новая синтагма
                self.sintagma_num += 1
                # Помечаем паузу у последнего слова
                if sentence_words:
                    sentence_words[-1]["has_pause"] = True
                    # Длительность паузы, если указана
                    if "time" in child.attrib:
                        sentence_words[-1]["pause"] = child.attrib["time"]

        # Постобработка: prev/next, sentence_len, fonetic_words_after
        for i in range(len(sentence_words)):
            sentence_words[i]["fonetic_words_after"] = (
                fonetic_words_count - sentence_words[i]["fonetic_words_before"] - 1
            )
            sentence_words[i]["sentence_len"] = words_count
            sentence_words[i]["prev_word"] = (
                sentence_words[i - 1]["word"] if i > 0 else None
            )
            sentence_words[i]["next_word"] = (
                sentence_words[i + 1]["word"] if i < len(sentence_words) - 1 else None
            )

        return sentence_words

    def parse_word(self, xml_word):
        word_info = {}

        # Исходная текстовая форма слова
        word_info["original"] = xml_word.attrib.get("original", "")

        # Есть ли признак фразового ударения
        word_info["nucleus"] = ("nucleus" in xml_word.attrib)

        # По флагам букв определяем, есть ли хотя бы одна заглавная
        word_info["is_capital"] = False

        letters = []
        allophones = []
        
        # Обрабатываем дочерние элементы <word>
        for child in xml_word:
            # Буква
            if child.tag == "letter":
                # Бит 16 (0b10000) указывает на заглавную букву
                if ("flag" in child.attrib) and (int(child.attrib["flag"]) & 16 > 0):
                    word_info["is_capital"] = True
                # Добавляем символ буквы
                letters.append(child.attrib["char"])
            elif child.tag == "allophone":
                # Аллофоны для train XML
                allophones.append(child.attrib["ph"])

        word_info["word"] = letters
        word_info["length"] = len(letters)
        word_info["allophones"] = allophones  # в test это будет []

        return word_info

    def check_if_fonetic(self, xml_word):
        # Фонетическое выделение по флагу буквы
        for child in xml_word:
            if (
                child.tag == "letter"
                and "flag" in child.attrib
                and (int(child.attrib["flag"]) & 1 > 0)
            ):
                return True
        return False

Создадим класс для построения словаря токенов и преобразования последовательностей токенов в числовые ID.

Используется для:
• кодирования букв (графем);
• кодирования аллофонов.

Поддерживает:
<PAD> — паддинг
<BOS> — начало последовательности
<EOS> — конец последовательности
<UNK> — неизвестный токен

In [22]:
class TokensEncoder:
    def __init__(self):
        # Начальный словарь служебных токенов
        self.tokens_count = 4
        self.tokens_to_ids = {
            "<PAD>": 0, # Для выравнивания последовательностей
            "<BOS>": 1, # Начало слова
            "<EOS>": 2, # Конец слова
            "<UNK>": 3 # Неизвестный токен
        }
    
    def fit(self, df_column):
        # Проходим по обучающей выборке и строим словарь токенов.
        for _, row in df_column.items():
            for token in row:
                # Если токена нет в словаре — добавляем
                if token not in self.tokens_to_ids:
                    self.tokens_to_ids[token] = self.tokens_count
                    self.tokens_count += 1
        # Обратный словарь id в токен
        self.ids_to_tokens = {v: k for k, v in self.tokens_to_ids.items()}

    def transform(self, df_column):
        # Преобразование списка токенов в список ID, используется при подготовке данных для обучения модели.
        return df_column.apply(
            lambda x: [
                self.tokens_to_ids.get(token, self.tokens_to_ids["<UNK>"])
                for token in x
            ]
        )

    def inverse_transform(self, df_column):
        # Обратное преобразование, используется при генерации результата модели.
        return df_column.apply(
            lambda x: [self.ids_to_tokens[token_id] for token_id in x]
        )

Отмечаем начало и конец каждой последовательности для работы с последовательными моделями. 
Используем специальные служебные токены:

<BOS> — Begin Of Sequence (начало слова);
<EOS> — End Of Sequence (конец слова).

В обучении модель получает вход с добавленными токенами <BOS>/<EOS>,  а при генерации предсказания <EOS> служит сигналом остановки.

In [23]:
def add_bos_eos(df_part, column_name):
    df_part[column_name] = df_part[column_name].apply(
        lambda x: ["<BOS>"] + x + ["<EOS>"]
    )

def remove_bos_eos_from_list(seq):
    # убираем BOS/EOS, если есть
    return [t for t in seq if t not in ["<BOS>", "<EOS>"]]

def remove_bos_eos(df_part, column_name):
    df_part[column_name] = df_part[column_name].apply(remove_bos_eos_from_list)

Обучим токенизатор на train и преобразуем последовательности train и val в ID-токены.

In [24]:
def train_and_apply_tokens_encoder(tokens_encoder, train_df, val_df, column):
    # Строим словарь токенов по тренировочным данным
    tokens_encoder.fit(train_df[column])
    # Преобразуем тренировочные данные в последовательности ID
    train_df[column] = tokens_encoder.transform(train_df[column])
    # Применяем токенизатор к валидационным данным
    val_df[column] = tokens_encoder.transform(val_df[column])

In [26]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cpu


Создаем класс набора данных для обучения модели "буквы → аллофоны".

На вход получает DataFrame, в котором каждая строка содержит:
• row["word"]       — список ID букв (после токенизации);
• row["allophones"] — список ID аллофонов (для train/val) или пустой список (для test).

Набор данных должен вернуть тензоры, пригодные для обработки в модели.

In [27]:
class WordsDataset(torch.utils.data.Dataset):
    def __init__(self, df):
        # сохраняем DataFrame с уже токенизированными последовательностями
        self.df = df

    def __len__(self):
        # количество примеров = количество строк в DataFrame
        return len(self.df)

    def __getitem__(self, index):
        # Возвращает одну обучающую пару (word, allophones) в виде LongTensor, будут дополнены паддингом в WordsPadder.
        row = self.df.iloc[index]
        # word и allophones – списки id (последовательности токенов)
        return (
            torch.LongTensor(row["word"]).to(device),
            torch.LongTensor(row["allophones"]).to(device),
        )

Реализуем класс для выравнивания (padding) входных и выходных последовательностей внутри батча.
Он выполняет следующие задачи:
• собрать список тензоров из Dataset;
• дополнить их паддингом до одинаковой длины;
• построить маски паддинга для внимания Transformer.

In [28]:
class WordsPadder:
    def __init__(self, pad_token):
        # Токен, используемый для дополнения последовательностей (<PAD>)
        self.pad_token = pad_token

    def __call__(self, batch):
        words = []
        allophone_sequences = []
        # Разделяем слова и аллофоны на отдельные списки
        for word, allophones in batch:
            words.append(word)
            allophone_sequences.append(allophones)
        # pad_sequence выравнивает тензоры по максимальной длине
        padded_words = torch.nn.utils.rnn.pad_sequence(
            words, padding_value=self.pad_token
        )
        # Маска паддинга: True там, где стоит паддинг - для Transformer
        words_pad_mask = (padded_words == self.pad_token)
        # Для выходных аллофонов
        padded_allophones = torch.nn.utils.rnn.pad_sequence(
            allophone_sequences, padding_value=self.pad_token
        )
        allophones_pad_mask = (padded_allophones == self.pad_token)

        # возвращаем (seq_len, batch_size) и маски (batch_size, seq_len)
        return (
            padded_words,
            padded_allophones,
            words_pad_mask.transpose(-1, -2),
            allophones_pad_mask.transpose(-1, -2),
        )

Используем нейросетевую архитектуру типа Seq2Seq на основе Transformer для решения задачи преобразования графем (букв) в аллофоны.

Архитектура включает:

• эмбеддинги входных токенов (букв);
• позиционные эмбеддинги для входа и выхода;
• Transformer (encoder–decoder) из PyTorch;
• линейный слой, превращающий выход декодера в распределение по словарю аллофонов.

В методе forward модель принимает выровненные батчи, а в word_to_allophones выполняет пошаговую генерацию до токена <EOS>.

In [29]:
class Seq2seq(torch.nn.Module):
    def __init__(
        self,
        word_tokens_count, # размер словаря букв
        allophone_tokens_count, # размер словаря аллофонов
        transformer_dim=512, # размерность эмбеддингов и скрытых слоёв
        max_seq_len=100, # максимальная длина генерируемой последовательности
        dropout_prob=0.1,
    ):
        super().__init__()
        self.max_seq_len = max_seq_len

        # Эмбеддинги входа (буквы)
        self.words_emb = torch.nn.Embedding(word_tokens_count, transformer_dim)
        self.words_positional_encoding = torch.nn.Embedding(max_seq_len, transformer_dim)

        # Эмбеддинги выхода (аллофоны)
        self.allophones_emb = torch.nn.Embedding(allophone_tokens_count, transformer_dim)
        self.allophones_positional_encoding = torch.nn.Embedding(max_seq_len, transformer_dim)
        # Дропаут для регуляризации
        self.dropout = torch.nn.Dropout(dropout_prob)

        # Основной Transformer
        # PyTorch реализует encoder-decoder внутри одного класса.
        self.transformer = torch.nn.Transformer(
            d_model=transformer_dim,
            dropout=dropout_prob
        )

        # Линейный слой проекции - переводит выход декодера в распределение по словарю аллофонов
        self.linear = torch.nn.Linear(transformer_dim, allophone_tokens_count)

    # Прямой проход, используется в обучении.
    def forward(self, words, allophones, words_pad_mask, allophones_pad_mask):
        """
        words: (S, N) последовательность букв
        allophones: (T, N) сдвинутая на 1 аллофонная последовательность для teacher forcing
        """
        S, N = words.shape
        T, _ = allophones.shape

        # Эмбеддинги и позиции для входа
        src = self.words_emb(words)
        pos_src = torch.arange(S, device=words.device)
        src = src + self.words_positional_encoding(pos_src).unsqueeze(1)
        src = self.dropout(src)

        # Эмбеддинги и позиции для выхода
        tgt = self.allophones_emb(allophones)
        pos_tgt = torch.arange(T, device=allophones.device)
        tgt = tgt + self.allophones_positional_encoding(pos_tgt).unsqueeze(1)
        tgt = self.dropout(tgt)

        # Маска, запрещающая смотреть в будущее при генерации
        tgt_mask = torch.nn.Transformer.generate_square_subsequent_mask(T).to(allophones.device)

        # Forward pass through Transformer
        out = self.transformer(
            src, tgt,
            tgt_mask=tgt_mask,
            src_key_padding_mask=words_pad_mask,
            tgt_key_padding_mask=allophones_pad_mask
        )  # (T, N, E)
        # Применяем линейный слой, logits по словарю аллофонов
        return self.linear(out)

    def word_to_allophones(self, src):
        """
        Авто-регрессивная генерация аллофонов.
        src — входное слово в виде тензора (S, 1)
        """
        self.eval() # отключаем dropout

        # Начинаем с <BOS> = 1
        tgt = torch.tensor([[1]], device=src.device).long()

        while True:
            S = src.size(0)
            T = tgt.size(0)

            src_mask = None
            # Маска для авто-регрессии (нельзя смотреть наперёд)
            tgt_mask = torch.nn.Transformer.generate_square_subsequent_mask(T).to(src.device)

            # Паддинг-маски (batch = 1)
            src_pad_mask = (src.squeeze(1) == 0).unsqueeze(0)    # (1, S)
            tgt_pad_mask = (tgt.squeeze(1) == 0).unsqueeze(0)    # (1, T)

            # Эмбеддинги + позиционные эмбеддинги
            src_e = self.words_emb(src) + self.words_positional_encoding(
                torch.arange(S, device=src.device)
            ).unsqueeze(1)

            tgt_e = self.allophones_emb(tgt) + self.allophones_positional_encoding(
                torch.arange(T, device=src.device)
            ).unsqueeze(1)

            # transformer forward
            out = self.transformer(
                src_e,
                tgt_e,
                tgt_mask=tgt_mask,
                src_key_padding_mask=src_pad_mask,
                tgt_key_padding_mask=tgt_pad_mask
            )
            # Предсказываем следующий токен по последнему таймстепу
            logits = self.linear(out)  # (T, 1, vocab)
            next_token = logits[-1, 0].argmax().item()

            # Добавляем к текущей последовательности
            tgt = torch.cat([tgt, torch.tensor([[next_token]], device=src.device)], dim=0)
            # Условия остановки
            if next_token == 2:  # <EOS>
                break
            if tgt.size(0) >= self.max_seq_len:
                break

        return tgt

Оценим качество модели, сравнивая предсказанные последовательности аллофонов с истинной (размеченной) последовательностью из тренировочного корпуса. 
Используем стандартные классификационные метрики и метрику расстояния Левенштейна, позволяющую измерять различие между двумя последовательностями (фонемами или аллофонами) на уровне замены, удаления и вставки токенов.

In [25]:
def levenshtein_distance(seq1, seq2):
    """
    Классический Левенштейн по спискам токенов (аллофоны / фонемы).
    Возвращает целое расстояние.
    """
    n, m = len(seq1), len(seq2)
    if n == 0:
        return m
    if m == 0:
        return n

    # dp[i][j] — расстояние между seq1[:i] и seq2[:j]
    dp = [[0] * (m + 1) for _ in range(n + 1)]

    for i in range(n + 1):
        dp[i][0] = i
    for j in range(m + 1):
        dp[0][j] = j

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = 0 if seq1[i - 1] == seq2[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,      # удаление
                dp[i][j - 1] + 1,      # вставка
                dp[i - 1][j - 1] + cost  # замена
            )

    return dp[n][m]

Функция pred_stage выполняет полный цикл оценки на валидационном наборе:
• получение предсказаний модели (авторегрессивная генерация);
• декодирование ID → токены;
• расчёт WRR (Word Recognition Rate) для аллофонов и фонем;
• расчёт нормированного Левенштейна;
• расчёт Accuracy, Precision, Recall, F1 по токенам;
• сравнение результатов с установленными порогами качества онлайн-теста.

Эти функции используются только на validation выборке, где доступны gold аллофоны. На тестовом наборе (Test_Phonetics.xml) метрика не считается, поскольку отсутствует истинная разметка.

In [40]:
def pred_stage(model, dl, word_encoder, allophone_encoder, stage_name="Evaluation"):
    print(f"\n {stage_name}")

    gold = [] # истинные аллофоны
    pred = [] # предсказанные моделью

    # Генерация предсказаний
    # batch_size должно быть 1 для авто-регрессивной генерации
    for words, allophones, words_pad_mask, allophones_pad_mask in tqdm(dl):
        # истинная последовательность с BOS/EOS
        gold_seq = allophones.view(-1).cpu().numpy()
        # предсказанная последовательность
        pred_seq = model.word_to_allophones(words).view(-1).cpu().numpy()

        gold.append(gold_seq)
        pred.append(pred_seq)

    # Декодирование ID в токены
    gold_dec = allophone_encoder.inverse_transform(pd.Series(gold))
    pred_dec = allophone_encoder.inverse_transform(pd.Series(pred))

    # удаляем служебные токены
    gold_dec = [seq[1:-1] for seq in gold_dec]
    pred_dec = [seq[1:-1] for seq in pred_dec]

    # WRR
    wrr_allophones = np.mean([gold_dec[i] == pred_dec[i] for i in range(len(gold_dec))])

    # WRR по фонемам
    def rm(a): return ''.join([c for c in a if not c.isdigit()])
    gold_ph = [[rm(x) for x in seq] for seq in gold_dec]
    pred_ph = [[rm(x) for x in seq] for seq in pred_dec]

    wrr_phonemes = np.mean([gold_ph[i] == pred_ph[i] for i in range(len(gold_ph))])

    # Нормированный Levenshtein
    levs = []
    for g, p in zip(gold_dec, pred_dec):
        if len(g)==0:
            continue
        lev = levenshtein_distance(g, p) / len(g)
        levs.append(lev)
    lev_mean = float(np.mean(levs))

    # Token metrics
    max_len = max(max(len(g), len(p)) for g, p in zip(gold, pred))
    gold_pad = np.array([list(g)+[0]*(max_len-len(g)) for g in gold])
    pred_pad = np.array([list(p)+[0]*(max_len-len(p)) for p in pred])

    accuracy = accuracy_score(gold_pad.ravel(), pred_pad.ravel())
    precision, recall, f1, _ = precision_recall_fscore_support(
        gold_pad.ravel(), pred_pad.ravel(),
        average='macro', zero_division=0
    )

    # Печать результатов
    print("\nMETRICS")
    print(f"WRR (аллофоны):        {wrr_allophones:.3f}")
    print(f"WRR (фонемы без удар): {wrr_phonemes:.3f}")
    print(f"Lev норм.:             {lev_mean:.3f}")
    print(f"Accuracy:              {accuracy:.3f}")
    print(f"Precision:             {precision:.3f}")
    print(f"Recall:                {recall:.3f}")
    print(f"F1-score:              {f1:.3f}")

    print("\nПОРОГИ")
    print("Фонемы ≥ 0.71:", "OK" if wrr_phonemes >= 0.71 else "FAIL")
    print("Аллофоны ≥ 0.55:", "OK" if wrr_allophones >= 0.55 else "FAIL")

    return {
        "wrr_allophones": wrr_allophones,
        "wrr_phonemes": wrr_phonemes,
        "lev": lev_mean,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

Загружаем обучающий XML-файл.

In [31]:
TRAIN_XML = "balzak_daughter.Result.xml" 

Извлекаем слова, их буквенные последовательности, аллофоны и дополнительные признаки с помощью ранее реализованного парсера.

In [32]:
xml = ET.parse(TRAIN_XML)
xml_root = xml.getroot()
parser = Parser()
df = pd.DataFrame(parser.parse_text(xml_root))
df.head()

,original,nucleus,is_capital,word,length,allophones,fonetic_words_before,word_num,has_pause,sintagma_num,fonetic_words_after,sentence_len,prev_word,next_word,pause
0,Графине,False,True,"[г, р, а, ф, и, н, е]",7,"[g, r, a1, f', i0, n', i4]",0,1,False,0,3,4,None,"[б, о, л, о, н, ь, и, н, и]",NaN
1,"Болоньини,",True,True,"[б, о, л, о, н, ь, и, н, и]",9,"[b, a2, l, a1, n', j, i0, n', i4]",1,2,True,0,2,4,"[г, р, а, ф, и, н, е]","[у, р, о, ж, д, ё, н, н, о, й]",115
2,урожденной,False,False,"[у, р, о, ж, д, ё, н, н, о, й]",10,"[u1, r, a1, zh, d', o0, n, a4, j]",2,3,False,1,1,4,"[б, о, л, о, н, ь, и, н, и]","[в, и, м, е, р, к, а, т, и]",NaN
3,Вимеркати,True,True,"[в, и, м, е, р, к, а, т, и]",9,"[v', i1, m', i1, r, k, a0, t', i4]",3,4,True,1,0,4,"[у, р, о, ж, д, ё, н, н, о, й]",None,645
4,Если,False,True,"[е, с, л, и]",4,"[j, e0, s, l', i4]",0,1,False,2,72,104,None,"[в, ы]",NaN


Подготовим данные для обучения, чтобы с ними могла работать модель Seq2seq.

In [33]:
# Фильтруем только те слова, у которых есть разметка аллофонов:
df = df[df["allophones"].apply(len) > 0].reset_index(drop=True)
len(df)

# Разбиваем на train/val:
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

# Добавляем BOS/EOS:
add_bos_eos(train_df, "word")
add_bos_eos(val_df, "word")
add_bos_eos(train_df, "allophones")
add_bos_eos(val_df, "allophones")

# Токенизируем буквы и аллофоны:
word_tokens_encoder = TokensEncoder()
train_and_apply_tokens_encoder(word_tokens_encoder, train_df, val_df, "word")

allophones_tokens_encoder = TokensEncoder()
train_and_apply_tokens_encoder(
    allophones_tokens_encoder, train_df, val_df, "allophones"
)

После подготовки обучающих и валидационных таблиц создаем объекты Dataset и DataLoader, которые будут подавать данные в модель.

• WordsDataset — собственный класс, возвращающий тензоры букв и аллофонов.
• DataLoader — автоматически формирует батчи, перемешивает данные для обучения и применяет collate_fn.

In [34]:
train_ds = WordsDataset(train_df)
val_ds = WordsDataset(val_df)

train_dl = torch.utils.data.DataLoader(
    train_ds,
    batch_size=42,
    shuffle=True,
    collate_fn=WordsPadder(word_tokens_encoder.tokens_to_ids["<PAD>"]),
)
val_dl = torch.utils.data.DataLoader(
    val_ds,
    batch_size=42,
    shuffle=False,
    collate_fn=WordsPadder(word_tokens_encoder.tokens_to_ids["<PAD>"]),
)

val_dl_for_pred = torch.utils.data.DataLoader(
    val_ds,
    batch_size=1,        
    shuffle=False,
    collate_fn=WordsPadder(word_tokens_encoder.tokens_to_ids["<PAD>"])
)

Обучение модели проводится в два этапа: train и validation.
Реализуем функцию train_stage, которая выполняет одну эпоху обработки батчей для обучения модели и для расчёта val_loss.

Так как последовательности аллофонов генерируются в режиме seq2seq, используется teacher forcing: на каждом шаге модели подаётся сдвинутая последовательность (allophones[:-1]), а целью является allophones[1:].

Функция train:
• запускает обучение модели на train DataLoader;
• оценивает её на val DataLoader (без градиентов);
• сохраняет лучшую модель по минимальному val_loss.

В качестве функции потерь используется CrossEntropyLoss с игнорированием паддинг-токенов, что позволяет корректно обучать модель на последовательностях разной длины.

In [35]:
def train_stage(model, optimizer, criterion, dl, stage_name, is_grad_needed=True):
    total_loss = 0.0
    print(stage_name + ":")
    for words, allophones, words_pad_mask, allophones_pad_mask in tqdm(dl):
        # для обучения обнуляем градиенты
        if is_grad_needed:
            optimizer.zero_grad()

        # сдвигаем таргеты на 1 (teacher forcing)
        outputs = model(
            words,
            allophones[:-1], # вход декодера
            words_pad_mask,
            allophones_pad_mask[:, :-1], # маска декодера для входа
        )  # (T-1, batch, vocab)

        # Считаем CrossEntropyLoss
        loss = criterion(
            outputs.view(-1, allophones_tokens_encoder.tokens_count),
            allophones[1:].reshape(-1),
        )

        # Обратное распространение ошибки
        if is_grad_needed:
            loss.backward()
            optimizer.step()

        total_loss += loss.item()

    # Средний loss за эпоху
    avg_loss = total_loss / len(dl)
    print("\tAvg loss:", avg_loss)
    return avg_loss


def train(model, optimizer, criterion, train_dl, val_dl, num_epochs=20):
    best_val = 1e9 # большое число, чтобы найти минимум
    for epoch in range(1, num_epochs + 1):
        print(f"Epoch {epoch}/{num_epochs}")
        model.train()
        train_stage(model, optimizer, criterion, train_dl, "Train", True)

        with torch.no_grad():
            model.eval()
            val_loss = train_stage(model, optimizer, criterion, val_dl, "Val", False)
            # сохраняем лучшую модель
            if val_loss < best_val:
                best_val = val_loss
                torch.save(model, "best_phonetics_model.pt")
                print("\tBest model saved with val_loss =", best_val)

In [36]:
model = Seq2seq(
    word_tokens_encoder.tokens_count,
    allophones_tokens_encoder.tokens_count,
    transformer_dim=512,
    max_seq_len=50,
).to(device)
# Оптимизатор
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = torch.nn.CrossEntropyLoss(
    ignore_index=allophones_tokens_encoder.tokens_to_ids["<PAD>"]
)

train(model, optimizer, criterion, train_dl, val_dl, num_epochs=20)

torch.save(model, "phonetics_model.pt")
with open("word_tokens_encoder.pickle", "wb") as f:
    pickle.dump(word_tokens_encoder, f)

with open("allophones_tokens_encoder.pickle", "wb") as f:
    pickle.dump(allophones_tokens_encoder, f)

/opt/anaconda3/lib/python3.13/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


Epoch 1/20
Train:


/opt/anaconda3/lib/python3.13/site-packages/torch/nn/functional.py:6041: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
100%|█████████████████████████████████████████| 668/668 [06:16<00:00,  1.78it/s]


	Avg loss: 1.3111291768665085
Val:


100%|█████████████████████████████████████████| 167/167 [00:18<00:00,  9.16it/s]


...

	Avg loss: 0.07354499719769297
	Best model saved with val_loss = 0.07354499719769297
Epoch 20/20
Train:


100%|█████████████████████████████████████████| 668/668 [06:27<00:00,  1.72it/s]


	Avg loss: 0.06637737597199436
Val:


100%|█████████████████████████████████████████| 167/167 [00:18<00:00,  9.13it/s]


	Avg loss: 0.22627573884175922


После завершения обучения оцениваем качество модели на валидационном наборе, где имеются истинные (gold) аллофоны. 

In [41]:
metrics = pred_stage(
    model,
    val_dl_for_pred,
    word_tokens_encoder,
    allophones_tokens_encoder,
    stage_name="Validation"
)


 Validation


/opt/anaconda3/lib/python3.13/site-packages/torch/nn/functional.py:6041: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
100%|███████████████████████████████████████| 7004/7004 [18:02<00:00,  6.47it/s]



METRICS
WRR (аллофоны):        0.742
WRR (фонемы без удар): 0.777
Lev норм.:             0.096
Accuracy:              0.982
Precision:             0.855
Recall:                0.877
F1-score:              0.864

ПОРОГИ
Фонемы ≥ 0.71: OK
Аллофоны ≥ 0.55: OK


Выполним инференс на тестовом корпусе, который содержит слова, но не содержит аллофонной разметки.

In [42]:
TEST_XML = "Test_Phonetics.xml"
OUTPUT_JSON = "Output_Phonetics.json"  

In [43]:
# Загружаем модель и токенизаторы:
loaded_model = torch.load("phonetics_model.pt", weights_only=False).to(device)
loaded_model.eval()

with open("word_tokens_encoder.pickle", "rb") as f:
    loaded_word_tokens_encoder = pickle.load(f)

with open("allophones_tokens_encoder.pickle", "rb") as f:
    loaded_allophones_tokens_encoder = pickle.load(f)


# Парсим Test_Phonetics.xml:
test_xml = ET.parse(TEST_XML)
test_root = test_xml.getroot()
test_parser = Parser()
test_df = pd.DataFrame(test_parser.parse_text(test_root))
test_df.head()

# На всякий случай убедимся, что allophones пустые
test_df["allophones"] = [[] for _ in range(len(test_df))]

# Добавим BOS/EOS к буквам и преобразуем в id:
add_bos_eos(test_df, "word")
test_df["word"] = loaded_word_tokens_encoder.transform(test_df["word"])

# Датасет и DataLoader (batch_size=1, потому что будем генерировать по одному слову):
test_ds = WordsDataset(test_df)
test_dl = torch.utils.data.DataLoader(
    test_ds,
    batch_size=1,
    shuffle=False,
    collate_fn=WordsPadder(loaded_word_tokens_encoder.tokens_to_ids["<PAD>"]),
)

# Генерируем аллофоны:
pred_list = []

with torch.no_grad():
    for word, allophones_dummy, words_pad_mask, allophones_pad_mask in tqdm(test_dl):
        # word: (S, 1)
        pred_allophones_ids = loaded_model.word_to_allophones(word).view(-1).cpu().numpy()
        pred_list.append(pred_allophones_ids)

pred_df = pd.DataFrame({"allophones": pred_list})
pred_df["allophones"] = loaded_allophones_tokens_encoder.inverse_transform(
    pred_df["allophones"]
)
remove_bos_eos(pred_df, "allophones")


# Соберём итоговый DataFrame с оригинальными словами и аллофонами:
pred_df["content"] = test_df["original"]

pred_df[["content", "allophones"]].head()

# Формирование JSON
words_records = json.loads(
    pred_df[["content", "allophones"]].to_json(
        orient="records",
        force_ascii=False
    )
)

result_json = [{"words": words_records}]

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(result_json, f, ensure_ascii=False, indent=4)

print("JSON сохранён:", OUTPUT_JSON)

/opt/anaconda3/lib/python3.13/site-packages/torch/nn/functional.py:6041: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
100%|███████████████████████████████████████| 3012/3012 [06:31<00:00,  7.70it/s]

JSON сохранён: Output_Phonetics.json
